In [ ]:
"""SESSION 20 - RTMDet-Ins-L: 5-fold CV + WBF ensemble (MMDetection).
"""
from __future__ import annotations
import multiprocessing
import os

# OOM prevention -- must be set BEFORE any torch import.
# PyTorch 2.4+ uses PYTORCH_CUDA_ALLOC_CONF; older versions used PYTORCH_ALLOC_CONF.
# Set both to be safe.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF",      "expandable_segments:True")
# Kaggle's notebook kernel may export matplotlib's inline backend. The isolated
# OpenMMLab subprocesses are headless and require a non-interactive backend.
os.environ["MPLBACKEND"] = "Agg"

import argparse
import json
import random
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml
SESSION_NAME   = "session20_rtmdet_ins_l"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASSES        = ["Nail_psoriasis", "SJS-TEN", "acne", "vitiligo"]
# ---------------------------------------------------------------------------
# Auto-install helpers
# ---------------------------------------------------------------------------
def _pip(*args):
    print(f"  [install] {' '.join(args)} ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *args,
    ])
# ---------------------------------------------------------------------------
# Isolated Python 3.10 environment for OpenMMLab
# ---------------------------------------------------------------------------
# Kaggle now ships Python 3.13 + torch 2.11/CUDA 12.8.  MMCV has NO prebuilt
# wheels for Python 3.13 or torch >= 2.5, and torch 2.4 has no cp313 wheel, so
# the OpenMMLab stack cannot be installed into Kaggle's own Python at all.
# Fix: build a separate Python 3.10 venv with `uv` (a few minutes, no compile)
# containing the known-good combo
#     torch 2.1.2 (cu121) + torchvision 0.16.2 + mmcv 2.1.0 + mmdet 3.3.0
# and re-run this whole script inside it.  Kaggle's own packages are untouched.
VENV_DIR   = Path(os.environ.get("S20_VENV", "/tmp/s20_mmdet_py310"))
VENV_PY    = VENV_DIR / "bin" / "python"
SCRIPT_OUT = Path("/kaggle/working/session20_rtmdet.py")
_PROBE = ("import pkg_resources; import torch, mmcv, mmengine, mmdet; "
          "from mmcv.ops import nms; "
          "print(torch.__version__, mmcv.__version__, mmengine.__version__, "
          "mmdet.__version__, 'cuda' if torch.cuda.is_available() else 'cpu')")
def _in_venv():
    try:
        return Path(sys.prefix).resolve() == VENV_DIR.resolve()
    except Exception:
        return False
def _venv_ok():
    if not VENV_PY.is_file():
        return False
    try:
        out = subprocess.check_output([str(VENV_PY), "-c", _PROBE], text=True,
                                      stderr=subprocess.STDOUT, timeout=300)
        print(f"  [deps] venv ready: {out.strip().splitlines()[-1]}")
        return True
    except Exception:
        return False
def _build_venv():
    print(f"  [deps] building isolated Python 3.10 env at {VENV_DIR} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
    uv = [sys.executable, "-m", "uv"]
    subprocess.check_call(uv + ["venv", "--python", "3.10", "--seed",
                                "--clear", str(VENV_DIR)])
    cons = VENV_DIR / "constraints.txt"
    cons.write_text("numpy<2\nopencv-python<4.11\nopencv-python-headless<4.11\n")
    pipi = uv + ["pip", "install", "--python", str(VENV_PY), "-c", str(cons)]
    # torch 2.1.2 on PyPI (Linux) is the CUDA 12.1 build -> works on T4/P100.
    subprocess.check_call(pipi + ["torch==2.1.2", "torchvision==0.16.2"])
    subprocess.check_call(pipi + [
        "numpy<2", "opencv-python-headless==4.10.0.84", "pyyaml", "pillow",
        "pycocotools", "scipy<1.14", "matplotlib", "shapely", "terminaltables",
        "six", "tqdm", "addict", "yapf", "termcolor", "rich", "packaging",
        "mmengine==0.10.4", "setuptools==75.1.0",
    ])
    subprocess.check_call(pipi + [
        "mmcv==2.1.0", "--only-binary", "mmcv",
        "--find-links", "https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html",
    ])
    subprocess.check_call(pipi + ["mmdet==3.3.0"])
    out = subprocess.check_output([str(VENV_PY), "-c", _PROBE], text=True)
    print(f"  [deps] verified: {out.strip()}")
def _script_source_path():
    """Path of this script as a file (works for `!python x.py` and for a
    notebook cell, where the cell source is recovered from IPython history)."""
    f = globals().get("__file__")
    if f and Path(f).is_file() and Path(f).suffix == ".py":
        return Path(f).resolve()
    try:
        from IPython import get_ipython
        src = get_ipython().user_ns["In"][-1]
        if "def _ensure_deps" in src:
            SCRIPT_OUT.parent.mkdir(parents=True, exist_ok=True)
            SCRIPT_OUT.write_text(src)
            print(f"  [deps] notebook cell saved to {SCRIPT_OUT}")
            return SCRIPT_OUT
    except Exception:
        pass
    raise SystemExit(
        "Cannot locate script source. Put `%%writefile session20.py` at the top "
        "of the cell, run it, then run `!python session20.py` in a new cell.")
def _ensure_deps():
    """Returns None when already inside the venv; otherwise builds it,
    re-runs this script inside it and returns the child's exit code."""
    if _in_venv():
        out = subprocess.check_output([sys.executable, "-c", _PROBE], text=True)
        print(f"  [deps] running inside venv: {out.strip()}")
        return None
    if not _venv_ok():
        _build_venv()
    script = _script_source_path()
    # Pass through CLI args when run as a script; in a notebook use S20_ARGS.
    if globals().get("__file__") and Path(globals()["__file__"]).is_file():
        extra = sys.argv[1:]
    else:
        extra = os.environ.get("S20_ARGS", "").split()
    print(f"  [deps] re-launching inside venv: {VENV_PY} {script} {' '.join(extra)}")
    # Stream child output line by line so it shows in the notebook cell too.
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen([str(VENV_PY), "-u", str(script), *extra], env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()
# ---------------------------------------------------------------------------
# GPU / device helpers
# ---------------------------------------------------------------------------
def _auto_device():
    try:
        import torch
        n = torch.cuda.device_count()
        if n >= 2:
            print(f"  [device] {n} GPUs -> 0,1 (DDP)")
            return "0,1"
        if n == 1:
            print("  [device] 1 GPU -> 0")
            return "0"
    except Exception:
        pass
    print("  [device] no GPU -> cpu")
    return "cpu"
def _single(device):
    return str(device).split(",")[0].strip() if device not in ("cpu", "") else "cpu"
def _n_gpus(device):
    return 0 if device in ("cpu", "") else len(str(device).split(","))
# ---------------------------------------------------------------------------
# Dataset root resolution (same logic as Session 18)
# ---------------------------------------------------------------------------
def _find_dataset_root(hint):
    if hint != "auto":
        p = Path(hint).resolve()
        if (p / "data.yaml").is_file():
            print(f"  [data-root] using explicit path: {p}")
            return p
        print(f"  [data-root] {p} not found, falling back to auto-detect")
    # Explicit Kaggle personal-dataset path (highest priority)
    _EXPLICIT = Path("/kaggle/input/datasets/munnamuntasirul/skin-disease-yolo/skin_disease_yolo")
    if _EXPLICIT.exists() and (_EXPLICIT / "data.yaml").is_file():
        print(f"  [auto-detect] found at explicit path: {_EXPLICIT}")
        return _EXPLICIT.resolve()
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for cand in sorted(kaggle_input.rglob("skin_disease_yolo")):
            if cand.is_dir() and (cand / "data.yaml").is_file():
                print(f"  [auto-detect] found named dataset at: {cand}")
                return cand.resolve()
        for dy in sorted(kaggle_input.rglob("data.yaml")):
            r = dy.parent
            if all((r / s / "images").is_dir() for s in ("train", "valid", "test")):
                print(f"  [auto-detect] found dataset layout at: {r}")
                return r.resolve()
    for cand in [Path("skin_disease_yolo"),
                 Path("../skin_disease_yolo"),
                 Path.home() / "skin_disease_yolo"]:
        cand = cand.resolve()
        if (cand / "data.yaml").is_file():
            print(f"  [auto-detect] found local dataset at: {cand}")
            return cand
    raise SystemExit(
        "\n=== SESSION 20 ERROR: skin_disease_yolo not found ===\n"
        "Fix A) On Kaggle: attach dataset zip as a Dataset input.\n"
        "Fix B) Anywhere:  pass --data-root /full/path/to/skin_disease_yolo\n"
        "=============================================================="
    )
# ---------------------------------------------------------------------------
# Dataset loading
# ---------------------------------------------------------------------------
def _gather_split(root, split):
    img_dir = root / split / "images"
    lbl_dir = root / split / "labels"
    if not img_dir.is_dir():
        return []
    items = []
    for img in sorted(img_dir.iterdir()):
        if img.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        lbl = lbl_dir / f"{img.stem}.txt"
        if lbl.is_file() and lbl.read_text().strip():
            items.append({"img": img, "stem": img.stem, "label": lbl})
    return items
# ---------------------------------------------------------------------------
# Class names: read from data.yaml (never trust a hard-coded list)
# ---------------------------------------------------------------------------
def _load_class_names(root):
    try:
        cfg = yaml.safe_load((root / "data.yaml").read_text()) or {}
    except Exception as e:
        print(f"  [classes] could not read data.yaml ({e}); using built-in list")
        return list(CLASSES)
    names = cfg.get("names")
    if isinstance(names, dict):                       # {0: 'acne', 1: ...}
        names = [str(names[k]) for k in sorted(names, key=lambda x: int(x))]
    elif isinstance(names, (list, tuple)):
        names = [str(n) for n in names]
    else:
        names = []
    nc = cfg.get("nc")
    if not names and nc:
        names = [f"class_{i}" for i in range(int(nc))]
    if not names:
        print("  [classes] data.yaml has no 'names'; using built-in list")
        return list(CLASSES)
    if nc and int(nc) != len(names):
        print(f"  [classes] WARNING: data.yaml nc={nc} but {len(names)} names")
    return names
def _scan_labels(items_by_split):
    """Count class ids and label formats so bad labels are reported, not crashed on."""
    ids, fmt = Counter(), Counter()
    for split, items in items_by_split.items():
        for it in items:
            for line in it["label"].read_text().splitlines():
                v = line.split()
                if not v:
                    continue
                try:
                    ids[int(float(v[0]))] += 1
                except ValueError:
                    fmt["unparseable"] += 1
                    continue
                n = len(v) - 1
                if n == 4:
                    fmt["bbox(5 values)"] += 1
                elif n >= 6 and n % 2 == 0:
                    fmt["polygon"] += 1
                else:
                    fmt["invalid"] += 1
    return ids, fmt
def _palette(n):
    base = [(220, 20, 60), (119, 11, 32), (0, 0, 142), (0, 0, 230),
            (106, 0, 228), (0, 60, 100), (0, 80, 100), (0, 0, 70),
            (0, 0, 192), (250, 170, 30), (100, 170, 30), (220, 220, 0),
            (175, 116, 175), (250, 0, 30), (165, 42, 42), (255, 77, 255)]
    return [base[i % len(base)] for i in range(n)]
# ---------------------------------------------------------------------------
# Stratified k-fold
# ---------------------------------------------------------------------------
def _multi_hot(items, n_classes):
    Y = np.zeros((len(items), n_classes), dtype=int)
    for i, it in enumerate(items):
        for line in it["label"].read_text().splitlines():
            if line.strip():
                c = int(float(line.split()[0]))
                if 0 <= c < n_classes:
                    Y[i, c] = 1
    return Y
def _stratified_kfold(Y, k, seed):
    """Assign a fold index [0, k) to every sample.
    Pure-numpy -- NO sklearn/scipy/iterstrat dependency.
    Why: Kaggle's Jupyter kernel caches whatever numpy was on disk at startup.
    A previous run that left numpy 1.x on disk causes the new kernel to load
    numpy 1.x into sys.modules. Even after pip upgrades numpy 2.x on disk,
    the in-process sys.modules still holds 1.x.  iterstrat/sklearn -> scipy
    -> from numpy import * then crashes: ImportError: _center.  By using only
    numpy (argmax/unique/where/RandomState work on both 1.x and 2.x) the fold
    split is always computed without touching scipy or sklearn.
    Strategy: stratify on dominant (argmax) class per image, shuffle per class,
    distribute round-robin across folds.
    """
    import numpy as np
    rng = np.random.RandomState(seed)
    dominant = Y.argmax(axis=1)
    n = len(dominant)
    fold_of = np.full(n, -1, dtype=int)
    for cls in np.unique(dominant):
        indices = np.where(dominant == cls)[0].copy()
        rng.shuffle(indices)
        for rank, idx in enumerate(indices):
            fold_of[idx] = int(rank % k)
    # Safety: assign any stray -1 round-robin (should never happen)
    for rank, idx in enumerate(np.where(fold_of == -1)[0]):
        fold_of[idx] = int(rank % k)
    return fold_of
def _repair_presence(fold_of, Y, k):
    K = Y.shape[1]
    members = [np.where(fold_of == f)[0] for f in range(k)]
    changed, guard = True, 0
    while changed and guard < 300:
        changed, guard = False, guard + 1
        for f in range(k):
            if len(members[f]) == 0:
                continue
            present = Y[members[f]].sum(axis=0)
            for c in range(K):
                if present[c] > 0:
                    continue
                best_src, best_idx = -1, -1
                for d in range(k):
                    if d == f:
                        continue
                    c_mask = Y[members[d], c] > 0
                    if int(c_mask.sum()) > 1:
                        best_src = d
                        best_idx = int(members[d][c_mask][0])
                        break
                if best_src == -1:
                    continue
                fold_of[best_idx] = f
                members[best_src] = members[best_src][members[best_src] != best_idx]
                members[f]        = np.append(members[f], best_idx)
                present[c]        = 1
                changed           = True
    return fold_of
# ---------------------------------------------------------------------------
# YOLO polygon labels -> COCO JSON converter
# ---------------------------------------------------------------------------
def _yolo_to_coco(items, names, dest):
    from PIL import Image as PImage
    images, annotations, ann_id = [], [], 1
    for img_id, it in enumerate(items, start=1):
        try:
            with PImage.open(it["img"]) as im:
                W, H = im.size
        except Exception:
            W, H = 640, 640
        images.append({"id": img_id, "file_name": str(it["img"].resolve()),
                        "width": W, "height": H})
        for line in it["label"].read_text().splitlines():
            v = line.strip().split()
            if len(v) == 5:
                # YOLO bbox line (cls cx cy w h) -> rectangle polygon
                cx, cy, bw, bh = (float(x) for x in v[1:])
                v = [v[0], cx - bw / 2, cy - bh / 2, cx + bw / 2, cy - bh / 2,
                     cx + bw / 2, cy + bh / 2, cx - bw / 2, cy + bh / 2]
            if len(v) < 7 or (len(v) - 1) % 2 != 0:
                continue
            try:
                c = int(float(v[0]))
            except ValueError:
                continue
            if not 0 <= c < len(names):
                continue
            coords = [float(x) for x in v[1:]]
            xs     = [min(1.0, max(0.0, coords[i])) * W for i in range(0, len(coords), 2)]
            ys     = [min(1.0, max(0.0, coords[i])) * H for i in range(1, len(coords), 2)]
            seg    = [x for pair in zip(xs, ys) for x in pair]
            area   = 0.5 * abs(sum(
                xs[i] * ys[(i + 1) % len(xs)] - xs[(i + 1) % len(xs)] * ys[i]
                for i in range(len(xs))))
            x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
            if area <= 1 or x1 <= x0 or y1 <= y0:
                continue
            annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": c + 1,
                "segmentation": [seg], "area": float(area),
                "bbox": [x0, y0, x1 - x0, y1 - y0], "iscrowd": 0,
            })
            ann_id += 1
    coco = {
        "images": images,
        "annotations": annotations,
        "categories": [{"id": i + 1, "name": n, "supercategory": "skin"}
                        for i, n in enumerate(names)],
    }
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(coco))
    print(f"  [coco] {len(images)} images, {len(annotations)} annotations -> {dest.name}")
# ---------------------------------------------------------------------------
# MMDetection config writer
# ---------------------------------------------------------------------------
def _write_config(cfg_path, train_json, val_json, work_dir, pretrained, args,
                  select_best=True):
    ng       = max(1, _n_gpus(args.device))
    total_b  = args.batch_per_gpu * ng
    # Scale LR from RTMDet-Ins-L default (lr=0.004 at bs=256)
    lr       = 0.004 * total_b / 256.0
    # Final model: validate only once at the end and keep the LAST checkpoint,
    # so the held-out test set is never used to pick a checkpoint.
    val_intv = max(200, args.max_iters // 10) if select_best else args.max_iters
    save_best = "'coco/segm_mAP_50'" if select_best else "None"
    warmup   = min(1000, args.max_iters // 10)
    cls_tuple = repr(tuple(CLASSES))
    if pretrained:
        load_line = f'load_from = "{pretrained}"'
    else:
        load_line = "# load_from = None  (train from scratch -- pretrained not downloaded)"
    cfg = f"""# SESSION 20 RTMDet-Ins-L config (auto-generated, do not edit manually)
_base_ = ['mmdet::rtmdet/rtmdet-ins_l_8xb32-300e_coco.py']
metainfo = dict(
    classes={cls_tuple},
    palette={_palette(len(CLASSES))},
)
# -- Training pipeline --------------------------------------------------------
train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
    # Keep mask geometry operations deterministic. CachedMosaic, RandomCrop,
    # and CachedMixUp can pass degenerate masks to cv2.warpAffine in MMDetection.
    dict(type='Resize', scale=(640, 640), keep_ratio=True),
    dict(type='RandomFlip', prob=0.5),
    dict(type='Pad', size=(640, 640), pad_val=dict(img=(114, 114, 114))),
    dict(type='PackDetInputs'),
]
test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(640, 640), keep_ratio=True),
    dict(type='Pad', size=(640, 640), pad_val=dict(img=(114, 114, 114))),
    dict(type='PackDetInputs',
         meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]
# -- Dataloaders ---------------------------------------------------------------
train_dataloader = dict(
    batch_size={args.batch_per_gpu}, num_workers=4, persistent_workers=True,  # per GPU
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(
        type='CocoDataset', metainfo=metainfo,
        data_root='', ann_file='{train_json}',
        data_prefix=dict(img=''), pipeline=train_pipeline,
    ),
)
val_dataloader = dict(
    batch_size=1, num_workers=2, persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset', metainfo=metainfo,
        data_root='', ann_file='{val_json}',
        data_prefix=dict(img=''), test_mode=True, pipeline=test_pipeline,
    ),
)
test_dataloader = val_dataloader
# _delete_=True: prevent base config's proposal_nums/format_only keys from merging
val_evaluator  = dict(_delete_=True, type='CocoMetric', ann_file='{val_json}',  metric=['segm'])
test_evaluator = dict(_delete_=True, type='CocoMetric', ann_file='{val_json}',  metric=['segm'])
# -- Model: N classes (from data.yaml) --
model = dict(bbox_head=dict(num_classes={len(CLASSES)}))
# -- Training schedule ---------------------------------------------------------
# _delete_=True: discard ALL keys inherited from the base config's train_cfg
# (e.g. max_epochs=300, dynamic_intervals) so only our iter-based keys remain.
# Without this, MMDet config merge passes max_epochs to IterBasedTrainLoop
# which does NOT accept it -> TypeError crash.
train_cfg = dict(_delete_=True, type='IterBasedTrainLoop',
                 max_iters={args.max_iters}, val_interval={val_intv})
val_cfg  = dict(_delete_=True, type='ValLoop')
test_cfg = dict(_delete_=True, type='TestLoop')
base_lr = {lr}
optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr=base_lr, weight_decay=0.05),
    clip_grad=dict(max_norm=35, norm_type=2),
)
param_scheduler = [
    dict(type='LinearLR', start_factor=1e-5, by_epoch=False,
         begin=0, end={warmup}),
    dict(type='CosineAnnealingLR', eta_min=base_lr * 0.05,
         begin={warmup}, end={args.max_iters}, T_max={args.max_iters}, by_epoch=False),
]
# EMAHook only.
# PipelineSwitchHook removed: MMDet 3.3.0 only accepts switch_epoch (not
# switch_iter). With IterBasedTrainLoop the hook signature is incompatible
# -> crashes at runner init. Mosaic stays active for all iters (minor
# quality trade-off vs a guaranteed crash).
custom_hooks = [
    dict(type='EMAHook', ema_type='ExpMomentumEMA', momentum=0.0002,
         update_buffers=True, priority=49),
]
default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=100),
    param_scheduler=dict(type='ParamSchedulerHook'),
    # MMEngine/CocoMetric reports metrics WITH 'coco/' prefix.
    # save_best MUST use the full key 'coco/segm_mAP_50' - bare 'segm_mAP_50'
    # causes KeyError at the first validation step.
    # MMEngine then writes the file as best_coco_segm_mAP_50_iter_*.pth.
    checkpoint=dict(type='CheckpointHook', interval=-1, by_epoch=False,
                    save_best={save_best}, rule='greater', max_keep_ckpts=1),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    visualization=dict(type='DetVisualizationHook'),
)
log_processor = dict(type='LogProcessor', window_size=50, by_epoch=False)
work_dir = '{work_dir}'
{load_line}
resume = False
"""
    cfg_path.write_text(cfg)
# ---------------------------------------------------------------------------
# Find the train/test scripts bundled in the mmdet wheel (no MIM required)
# ---------------------------------------------------------------------------
def _get_mmdet_tool(name):
    try:
        import mmdet
        root = Path(mmdet.__file__).parent
        for candidate in [
            root / ".mim" / "tools" / f"{name}.py",
            root / "tools" / f"{name}.py",
            root.parent / "tools" / f"{name}.py",
            root.parent.parent / "tools" / f"{name}.py",
        ]:
            if candidate.is_file():
                print(f"  [tools] found {name}.py: {candidate}")
                return str(candidate)
    except Exception:
        pass
    return None
def _get_train_script():
    return _get_mmdet_tool("train")
def _get_test_script():
    return _get_mmdet_tool("test")
# ---------------------------------------------------------------------------
# Download pretrained checkpoint
# ---------------------------------------------------------------------------
def _download_pretrained(work):
    ckpt_dir = work / "pretrained"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    candidates = list(ckpt_dir.glob("rtmdet-ins_l*.pth"))
    if candidates:
        print(f"  [pretrained] already exists: {candidates[0]}")
        return str(candidates[0])
    checkpoint = ckpt_dir / "rtmdet-ins_l_8xb32-300e_coco.pth"
    url = (
        "https://download.openmmlab.com/mmdetection/v3.0/rtmdet/"
        "rtmdet-ins_l_8xb32-300e_coco/"
        "rtmdet-ins_l_8xb32-300e_coco_20221124_103237-78d1d652.pth"
    )
    print("  [pretrained] downloading RTMDet-Ins-L checkpoint directly ...")
    try:
        from urllib.request import urlretrieve
        urlretrieve(url, checkpoint)
        if checkpoint.is_file() and checkpoint.stat().st_size > 1_000_000:
            return str(checkpoint)
        raise RuntimeError("downloaded checkpoint is unexpectedly small")
    except Exception as e:
        checkpoint.unlink(missing_ok=True)
        print(f"  [warn] checkpoint download failed ({e}); will train from scratch")
    return ""
# ---------------------------------------------------------------------------
# Training launcher
# FIX: CUDA_VISIBLE_DEVICES must be set explicitly in the subprocess env.
# Without it, torch.distributed rank 1 calls set_device(1) and crashes with
# 'CUDA error: invalid device ordinal' whenever only 1 device is visible
# inside the child process (a common Kaggle DDP pitfall across folds).
# FIX: Re-probe actual GPU count in a fresh child process to guard against
# NCCL/CUDA state left over from a previous fold's DDP group.
# FIX: Fresh random port per fold prevents NCCL rendezvous port collisions.
# ---------------------------------------------------------------------------
def _actual_gpu_count(cuda_visible: str) -> int:
    """Query device count inside a fresh subprocess with the given CUDA env."""
    try:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = cuda_visible
        out = subprocess.check_output(
            [sys.executable, "-c",
             "import torch; print(torch.cuda.device_count())"],
            env=env, text=True, timeout=30,
        ).strip()
        return int(out)
    except Exception:
        return 1
def _run_train(cfg_path, device, work_dir):
    import time
    os.environ["MPLBACKEND"] = "Agg"
    tool = _get_train_script()
    if tool is None:
        raise RuntimeError(
            "MMDetection's bundled tools/train.py was not found after installation"
        )
    # Build the CUDA_VISIBLE_DEVICES string from the requested device spec.
    if device in ("cpu", ""):
        cuda_visible = ""
    elif device == "0,1":
        cuda_visible = "0,1"
    else:
        cuda_visible = device  # e.g. "0"
    # Re-probe the actual count inside a child process with the same env.
    # This catches cases where a previous fold's DDP run left NCCL state that
    # makes device 1 unreachable for a new process group.
    if cuda_visible and cuda_visible != "":
        visible_count = _actual_gpu_count(cuda_visible)
    else:
        visible_count = 0
    # Use a fresh random port to avoid port collisions across folds.
    port = random.randint(20000, 49999)
    # Build subprocess environment with CUDA_VISIBLE_DEVICES always set.
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    if cuda_visible:
        env["CUDA_VISIBLE_DEVICES"] = cuda_visible
    if visible_count >= 2:
        print(f"  [train] DDP on {visible_count} GPUs "
              f"(CUDA_VISIBLE_DEVICES={cuda_visible}), port={port} ...")
        cmd = [
            sys.executable, "-m", "torch.distributed.run",
            "--nproc_per_node", str(visible_count),
            "--master_port", str(port),
            tool, str(cfg_path),
            "--work-dir", work_dir,
            "--launcher", "pytorch",
        ]
    elif visible_count == 1:
        print(f"  [train] single GPU "
              f"(CUDA_VISIBLE_DEVICES={cuda_visible}) via tools/train.py ...")
        cmd = [sys.executable, tool, str(cfg_path), "--work-dir", work_dir]
    else:
        print("  [train] CPU training (no CUDA device found) ...")
        cmd = [sys.executable, tool, str(cfg_path), "--work-dir", work_dir]
    subprocess.check_call(cmd, env=env)
    # Brief pause after each fold to allow NCCL rendezvous sockets and
    # CUDA contexts to be released before the next fold's DDP group starts.
    time.sleep(5)
def _best_checkpoint(work_dir):
    wd = Path(work_dir)
    # MMEngine writes checkpoint as best_coco_segm_mAP_50_iter_*.pth
    # when save_best='coco/segm_mAP_50' (slashes -> underscores in filename)
    candidates = list(wd.glob("best_coco_segm_mAP_50_iter_*.pth"))
    if candidates:
        return str(max(candidates, key=lambda p: p.stat().st_mtime))
    fallback = sorted(wd.glob("iter_*.pth"), key=lambda p: p.stat().st_mtime)
    return str(fallback[-1]) if fallback else None
# ---------------------------------------------------------------------------
# Test config (FIX 3: format_only=True, FIX 4: clean filename)
# ---------------------------------------------------------------------------
def _make_test_cfg(orig_cfg_path, test_json, pred_prefix):
    """
    Append test overrides to original config.
    Key fix: format_only=True + outfile_prefix -> saves {pred_prefix}.segm.json
    in COCO detection format that pycocotools can load directly.
    Using --out (pickle) would save wrong format -> all mAP = 0.
    """
    cls_tuple = repr(tuple(CLASSES))
    override = f"""
# -- test-split override (auto-injected by SESSION 20) ------------------------
# _delete_=True: fully replace the training config's test_dataloader/evaluator
# so no stale keys (batch_sampler, filter_cfg, format_only=False) leak in.
test_dataloader = dict(
    _delete_=True,
    batch_size=1,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset',
        metainfo=dict(
            classes={cls_tuple},
            palette={_palette(len(CLASSES))},
        ),
        ann_file='{test_json}',
        data_root='',
        data_prefix=dict(img=''),
        test_mode=True,
        pipeline=[
            dict(type='LoadImageFromFile'),
            dict(type='Resize', scale=(640, 640), keep_ratio=True),
            dict(type='Pad', size=(640, 640), pad_val=dict(img=(114, 114, 114))),
            dict(type='PackDetInputs',
                 meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
        ],
    ),
)
# format_only=True: saves COCO-format JSON at {{outfile_prefix}}.segm.json
# _delete_=True: discard format_only=False that came from training config val_evaluator
test_evaluator = dict(
    _delete_=True,
    type='CocoMetric',
    ann_file='{test_json}',
    metric=['segm'],
    format_only=True,
    outfile_prefix='{pred_prefix}',
)
"""
    # FIX 4: use a clean .py filename, not ambiguous .test.py
    test_cfg_path = orig_cfg_path.parent / f"test_{orig_cfg_path.stem}.py"
    test_cfg_path.write_text(orig_cfg_path.read_text() + override)
    return test_cfg_path
def _run_test(cfg_path, checkpoint, pred_prefix, test_json, device, work_dir):
    """
    Run MMDet test with format_only=True.
    Predictions are saved to {pred_prefix}.segm.json in COCO format.
    Returns the path to the saved JSON.
    """
    test_cfg = _make_test_cfg(cfg_path, test_json, pred_prefix)
    test_script = _get_test_script()
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    if _single(device) != "cpu":
        env["CUDA_VISIBLE_DEVICES"] = _single(device)
    if test_script:
        cmd = [
            sys.executable, test_script,
            str(test_cfg), checkpoint,
            "--work-dir", work_dir,
        ]
    else:
        raise RuntimeError(
            "MMDetection's bundled tools/test.py was not found after installation"
        )
    subprocess.check_call(cmd, env=env)
    return Path(f"{pred_prefix}.segm.json")




















def _coco_eval(gt_json, dt_json, names):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    dt_path = Path(dt_json)
    if not dt_path.is_file() or dt_path.stat().st_size <= 2:
        print(f"  [eval] {dt_path.name} empty or missing -> mAP50 = 0.0")
        return {"mAP50": 0.0, "mAP50_95": 0.0, "per_class_mAP50": {}}
    data = json.loads(dt_path.read_text())
    if not data:
        return {"mAP50": 0.0, "mAP50_95": 0.0, "per_class_mAP50": {}}
    coco_gt = COCO(str(gt_json))
    coco_dt = coco_gt.loadRes(str(dt_json))
    ev = COCOeval(coco_gt, coco_dt, "segm")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    per_class = {}
    for ci, name in enumerate(names, start=1):
        e = COCOeval(coco_gt, coco_dt, "segm")
        e.params.catIds = [ci]
        e.evaluate(); e.accumulate(); e.summarize()
        per_class[name] = float(e.stats[1])
    return {"mAP50": float(ev.stats[1]), "mAP50_95": float(ev.stats[0]),
            "per_class_mAP50": per_class}
# ---------------------------------------------------------------------------
# WBF for COCO segm prediction JSONs
# ---------------------------------------------------------------------------
def _iou_xywh(a, b):
    """IoU for COCO bbox format [x, y, w, h]."""
    ax1, ay1 = a[0], a[1]; ax2, ay2 = a[0]+a[2], a[1]+a[3]
    bx1, by1 = b[0], b[1]; bx2, by2 = b[0]+b[2], b[1]+b[3]
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    ua = (ax2-ax1)*(ay2-ay1); ub = (bx2-bx1)*(by2-by1)
    union = ua + ub - inter
    return inter / union if union > 0 else 0.0
def _merge_fold_predictions(fold_pred_jsons, out_json, wbf_iou=0.55, min_support=1):
    """
    WBF merge across fold prediction files.
    Each file is a COCO segm prediction JSON: list of
    {image_id, category_id, bbox [x,y,w,h], score, segmentation}.
    """
    by_image = defaultdict(lambda: [[] for _ in fold_pred_jsons])
    for fi, fp in enumerate(fold_pred_jsons):
        if not Path(fp).is_file():
            continue
        for d in json.loads(Path(fp).read_text()):
            by_image[d["image_id"]][fi].append(d)
    n_models = len(fold_pred_jsons)
    merged   = []
    for img_id, per_model in by_image.items():
        flat = [d for dets in per_model for d in dets]
        if not flat:
            continue
        flat.sort(key=lambda d: d["score"], reverse=True)
        used = [False] * len(flat)
        for i in range(len(flat)):
            if used[i]:
                continue
            cluster = [flat[i]]; used[i] = True
            for j in range(i + 1, len(flat)):
                if used[j] or flat[j]["category_id"] != flat[i]["category_id"]:
                    continue
                if _iou_xywh(flat[i]["bbox"], flat[j]["bbox"]) >= wbf_iou:
                    cluster.append(flat[j]); used[j] = True
            if len(cluster) < min_support:
                continue
            w     = np.array([d["score"] for d in cluster])
            boxes = np.array([d["bbox"] for d in cluster])
            fbox  = (boxes * w[:, None]).sum(0) / w.sum()
            score = float(min(1.0, w.sum() / n_models))
            best  = cluster[0]
            merged.append({
                "image_id":    img_id,
                "category_id": best["category_id"],
                "bbox":        fbox.tolist(),
                "score":       score,
                "segmentation": best.get("segmentation", []),
            })
    Path(out_json).write_text(json.dumps(merged))
    print(f"  [wbf] {n_models} folds -> {len(merged)} merged detections")
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(
        description="SESSION 20 -- RTMDet-Ins-L 5-fold CV on skin_disease_yolo."
    )
    ap.add_argument("--data-root",     default="auto",
                    help="'auto'=scan Kaggle input; or pass explicit path")
    ap.add_argument("--work-dir",      default=f"/kaggle/working/{SESSION_NAME}")
    ap.add_argument("--folds",         type=int,   default=5)
    ap.add_argument("--max-iters",     type=int,   default=3800,
                    help="Iterations per fold. 3800x5 folds ~ 7.5h + setup fits "
                         "safely in the 12h Kaggle limit. (8000 timed out at fold1/iter3900.)")
    ap.add_argument("--batch-per-gpu", type=int,   default=2,
                    help="Per-GPU batch. 2/GPU=4 total on T4x2. "
                         "4/GPU causes OOM on 15 GiB T4s."
                         "Batch=2 keeps peak GPU memory under 13 GiB.")
    ap.add_argument("--device",        default="auto")
    ap.add_argument("--cv-seed",       type=int,   default=42)
    ap.add_argument("--wbf-iou",       type=float, default=0.55)
    ap.add_argument("--min-support",   type=int,   default=1)
    ap.add_argument("--skip-existing", action="store_true", default=True)
    ap.add_argument("--no-skip-existing", dest="skip_existing", action="store_false")
    ap.add_argument("--skip-train",    action="store_true",
                    help="Skip training; load existing checkpoints and run Phase B.")
    ap.add_argument("--no-final",      action="store_true",
                    help="Skip Phase D (final all-trainval model) to save ~1.5-2h.")
    ap.add_argument("--quick",         action="store_true",
                    help="Smoke test: 200 iters, 200 trainval, 50 test images.")
    args, _ = ap.parse_known_args()
    multiprocessing.freeze_support()
    print(f"  [mem] PYTORCH_ALLOC_CONF = {os.environ.get('PYTORCH_ALLOC_CONF','not set')}")
    _rc_child = _ensure_deps()
    if _rc_child is not None:      # work was done inside the venv child
        return _rc_child
    if args.device == "auto":
        args.device = _auto_device()
    print(f"  [device] {args.device}  ({_n_gpus(args.device)} GPU(s))")
    global CLASSES
    root  = _find_dataset_root(args.data_root)
    names = _load_class_names(root)
    print(f"\nDataset root : {root}")
    train_items = _gather_split(root, "train")
    valid_items = _gather_split(root, "valid")
    test_items  = _gather_split(root, "test")
    trainval    = train_items + valid_items
    print(f"Train        : {len(train_items)} images")
    print(f"Valid        : {len(valid_items)} images")
    print(f"Test         : {len(test_items)} images")
    print(f"Trainval     : {len(trainval)} images  (pooled for CV)")
    ids, fmt = _scan_labels({"train": train_items, "valid": valid_items,
                             "test": test_items})
    print(f"Label formats: {dict(fmt)}")
    print(f"Class ids    : {dict(sorted(ids.items()))}")
    if ids and max(ids) >= len(names):
        print(f"  [classes] WARNING: labels use class id {max(ids)} but data.yaml "
              f"lists only {len(names)} names -> adding placeholder names. "
              "Fix data.yaml 'names' to get real class names.")
        names += [f"class_{i}" for i in range(len(names), max(ids) + 1)]
    if ids and min(ids) < 0:
        print("  [classes] WARNING: negative class ids found; those lines are ignored")
    if fmt.get("bbox(5 values)"):
        print(f"  [labels] WARNING: {fmt['bbox(5 values)']} bbox-only lines -> "
              "converted to rectangle masks (segmentation quality will be lower)")
    CLASSES = list(names)
    K = len(names)
    print(f"Classes used ({K}): {names}")
    if args.quick:
        random.Random(0).shuffle(trainval)
        trainval   = trainval[:200]
        test_items = test_items[:50]
        args.max_iters = 200
        print("  [quick] smoke-test mode: 200 iters, 200 trainval, 50 test")
    if args.quick:
        args.work_dir = args.work_dir.rstrip("/") + "_quick"
    work = Path(args.work_dir).resolve()
    (work / "coco_json").mkdir(parents=True, exist_ok=True)
    (work / "ensemble").mkdir(parents=True, exist_ok=True)
    pretrained = _download_pretrained(work)
    Y       = _multi_hot(trainval, K)
    fold_of = _repair_presence(
        _stratified_kfold(Y, args.folds, args.cv_seed), Y, args.folds)
    print(f"\nFold val sizes: { {f: int((fold_of==f).sum()) for f in range(args.folds)} }")
    for f in range(args.folds):
        val_idx = np.where(fold_of == f)[0]
        cls_cnt = Counter(
            int(float(line.split()[0]))
            for idx in val_idx
            for line in trainval[idx]["label"].read_text().splitlines()
            if line.strip()
        )
        fold_counts = {
            (names[c] if 0 <= c < K else f"id{c}"): cnt
            for c, cnt in sorted(cls_cnt.items())
        }
        print(f"  fold {f} val: {fold_counts}")
    summary = {
        "session":    SESSION_NAME,
        "model":      "RTMDet-Ins-L",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset":    str(root),
        "dataset_sizes": {
            "train": len(train_items), "valid": len(valid_items), "test": len(test_items),
        },
        "config": {
            "folds": args.folds, "max_iters": args.max_iters,
            "batch_per_gpu": args.batch_per_gpu, "device": args.device,
            "cv_seed": args.cv_seed, "wbf_iou": args.wbf_iou,
        },
        "classes": names,
        "phaseA_cv": [], "phaseB_ensemble": {}, "phaseD_final": {},
    }
    summary_path = work / f"{SESSION_NAME}_summary.json"
    def _save():
        summary_path.write_text(json.dumps(summary, indent=2, default=str))
    # Convert test split to COCO JSON once
    test_gt_json = work / "coco_json" / "test_gt.json"
    if not test_gt_json.is_file():
        print("\nConverting test split to COCO JSON ...")
        _yolo_to_coco(test_items, names, test_gt_json)
    # ==========================================================================
    # PHASE A -- 5-fold cross-validation
    # ==========================================================================
    print("\n" + "="*64 + "\nPHASE A -- 5-fold CV  (RTMDet-Ins-L)\n" + "="*64)
    fold_checkpoints = []
    fold_cfg_paths   = []
    if not args.skip_train:
        for f in range(args.folds):
            print(f"\n--- Fold {f}/{args.folds-1} ---")
            val_idx = np.where(fold_of == f)[0]
            tr_idx  = np.where(fold_of != f)[0]
            fold_dir  = work / "folds" / f"fold_{f}"
            fold_work = work / "runs"  / f"fold_{f}"
            fold_dir.mkdir(parents=True, exist_ok=True)
            fold_work.mkdir(parents=True, exist_ok=True)
            tr_json = work / "coco_json" / f"fold_{f}_train.json"
            va_json = work / "coco_json" / f"fold_{f}_val.json"
            if not tr_json.is_file():
                _yolo_to_coco([trainval[i] for i in tr_idx], names, tr_json)
            if not va_json.is_file():
                _yolo_to_coco([trainval[i] for i in val_idx], names, va_json)
            cfg_path = fold_dir / "rtmdet_fold.py"
            best_pt  = _best_checkpoint(str(fold_work))
            if args.skip_existing and best_pt:
                print(f"  [skip] fold {f}: checkpoint -> {best_pt}")
            else:
                _write_config(cfg_path, str(tr_json), str(va_json),
                              str(fold_work), pretrained, args)
                print(f"  [train] {len(tr_idx)} train, {len(val_idx)} val, "
                      f"{args.max_iters} iters ...")
                _run_train(cfg_path, args.device, str(fold_work))
                best_pt = _best_checkpoint(str(fold_work))
            if not best_pt:
                print(f"  [warn] fold {f}: no checkpoint found, skipping")
                continue
            fold_checkpoints.append(best_pt)
            fold_cfg_paths.append(cfg_path)
            # Validate: run test on val split, save predictions, eval with pycocotools
            print(f"  [eval] fold {f} val ...")
            va_prefix = str(work / "ensemble" / f"fold_{f}_val")
            try:
                pred_json = _run_test(cfg_path, best_pt, va_prefix, str(va_json),
                                       args.device, str(fold_work))
                m = _coco_eval(va_json, pred_json, names)
                summary["phaseA_cv"].append({
                    "fold": f, "checkpoint": best_pt,
                    "val_mAP50": m["mAP50"], "val_mAP50_95": m["mAP50_95"],
                    "per_class_mAP50": m["per_class_mAP50"],
                })
                print(f"  Fold {f}: val mAP50={m['mAP50']:.4f}  mAP50-95={m['mAP50_95']:.4f}")
            except Exception as e:
                print(f"  [warn] fold {f} eval failed: {e}")
            _save()
        if summary["phaseA_cv"]:
            arr = np.array([r["val_mAP50"] for r in summary["phaseA_cv"]], dtype=float)
            summary["phaseA_cv_mean_mAP50"] = float(np.nanmean(arr))
            summary["phaseA_cv_std_mAP50"]  = float(np.nanstd(arr))
            print(f"\nCV mAP50 = {np.nanmean(arr):.4f} +/- {np.nanstd(arr):.4f}")
            _save()
    else:
        # Load existing checkpoints
        for f in range(args.folds):
            fold_dir  = work / "folds" / f"fold_{f}"
            fold_work = work / "runs"  / f"fold_{f}"
            ckpt = _best_checkpoint(str(fold_work))
            cfg  = fold_dir / "rtmdet_fold.py"
            if ckpt and cfg.is_file():
                fold_checkpoints.append(ckpt)
                fold_cfg_paths.append(cfg)
        print(f"--skip-train: loaded {len(fold_checkpoints)} checkpoints")
    # ==========================================================================
    # PHASE B -- WBF ensemble on held-out test
    # ==========================================================================
    if test_items and fold_checkpoints:
        print("\n" + "="*64 + "\nPHASE B -- WBF ensemble on test\n" + "="*64)
        fold_pred_jsons = []
        single_scores   = []
        for fi, (ckpt, cfg_path) in enumerate(zip(fold_checkpoints, fold_cfg_paths)):
            print(f"  predicting test with fold {fi} ...")
            pred_prefix = str(work / "ensemble" / f"fold_{fi}_test")
            try:
                pred_json = _run_test(cfg_path, ckpt, pred_prefix, str(test_gt_json),
                                       args.device, str(work / "runs" / f"fold_{fi}"))
                m = _coco_eval(test_gt_json, pred_json, names)
                single_scores.append(m["mAP50"])
                fold_pred_jsons.append(pred_json)
                print(f"    fold {fi} solo test mAP50 = {m['mAP50']:.4f}")
            except Exception as e:
                print(f"  [warn] fold {fi} test failed: {e}")
        if fold_pred_jsons:
            ens_dt = work / "ensemble" / "ensemble_test.json"
            _merge_fold_predictions(fold_pred_jsons, ens_dt, args.wbf_iou, args.min_support)
            ens    = _coco_eval(test_gt_json, ens_dt, names)
            mean_s = float(np.mean(single_scores)) if single_scores else None
            summary["phaseB_ensemble"] = {
                "solo_fold_test_mAP50": single_scores,
                "solo_mean_test_mAP50": mean_s,
                "ensemble_test_mAP50":  ens["mAP50"],
                "ensemble_mAP50_95":    ens["mAP50_95"],
                "per_class_mAP50":      ens["per_class_mAP50"],
                "ensemble_gain": (ens["mAP50"] - mean_s) if mean_s is not None else None,
            }
            print(f"\n  Solo-mean test mAP50 = {mean_s:.4f}")
            print(f"  ENSEMBLE test mAP50  = {ens['mAP50']:.4f}  "
                  f"(gain {ens['mAP50']-(mean_s or 0):+.4f})")
            print(f"  mAP50-95             = {ens['mAP50_95']:.4f}")
            print("  Per-class mAP50:")
            for cls_name, v in ens["per_class_mAP50"].items():
                print(f"    {cls_name}: {v:.4f}")
            _save()
    # ==========================================================================
    # PHASE D -- final model on all trainval
    # ==========================================================================
    if not args.skip_train and not args.no_final and test_items:
        print("\n" + "="*64 + "\nPHASE D -- final model on all trainval\n" + "="*64)
        print(f"  Training on all {len(trainval)} trainval images ...")
        final_dir  = work / "folds" / "final"
        final_work = work / "runs"  / "final_all_trainval"
        final_dir.mkdir(parents=True, exist_ok=True)
        final_work.mkdir(parents=True, exist_ok=True)
        all_tr_json = work / "coco_json" / "final_train.json"
        all_va_json = work / "coco_json" / "final_val.json"
        if not all_tr_json.is_file():
            _yolo_to_coco(trainval,   names, all_tr_json)
        if not all_va_json.is_file():
            _yolo_to_coco(test_items, names, all_va_json)
        cfg_path   = final_dir / "rtmdet_final.py"
        final_best = _best_checkpoint(str(final_work))
        if args.skip_existing and final_best:
            print(f"  [skip] final checkpoint: {final_best}")
        else:
            _write_config(cfg_path, str(all_tr_json), str(all_va_json),
                          str(final_work), pretrained, args, select_best=False)
            print(f"  [train] {len(trainval)} images, {args.max_iters} iters ...")
            _run_train(cfg_path, args.device, str(final_work))
            final_best = _best_checkpoint(str(final_work))
        if final_best:
            print("  [eval] evaluating final model on held-out test ...")
            pred_prefix = str(work / "ensemble" / "final_test")
            try:
                pred_json = _run_test(cfg_path, final_best, pred_prefix,
                                       str(test_gt_json), args.device, str(final_work))
                fe = _coco_eval(test_gt_json, pred_json, names)
                summary["phaseD_final"] = {
                    "checkpoint":      final_best,
                    "trainval_size":   len(trainval),
                    "test_mAP50":      fe["mAP50"],
                    "test_mAP50_95":   fe["mAP50_95"],
                    "per_class_mAP50": fe["per_class_mAP50"],
                }
                print(f"\n  FINAL MODEL mAP50    = {fe['mAP50']:.4f}")
                print(f"  FINAL MODEL mAP50-95 = {fe['mAP50_95']:.4f}")
                for cls_name, v in fe["per_class_mAP50"].items():
                    print(f"    {cls_name}: {v:.4f}")
            except Exception as e:
                print(f"  [warn] final eval failed: {e}")
            _save()
    # -- final report ----------------------------------------------------------
    _save()
    print("\n" + "="*64 + "\nSESSION 20 COMPLETE\n" + "="*64)
    if "phaseA_cv_mean_mAP50" in summary:
        print(f"CV (val) mAP50        : {summary['phaseA_cv_mean_mAP50']:.4f} "
              f"+/- {summary['phaseA_cv_std_mAP50']:.4f}")
    if summary["phaseB_ensemble"]:
        b = summary["phaseB_ensemble"]
        if b.get("solo_mean_test_mAP50") is not None:
            print(f"5-fold solo mean mAP50: {b['solo_mean_test_mAP50']:.4f}")
        print(f"5-fold ENSEMBLE mAP50 : {b['ensemble_test_mAP50']:.4f}  "
              f"(gain {b.get('ensemble_gain', 0):+.4f})")
        print(f"5-fold mAP50-95       : {b['ensemble_mAP50_95']:.4f}")
    if summary.get("phaseD_final"):
        d = summary["phaseD_final"]
        print(f"FINAL MODEL mAP50     : {d['test_mAP50']:.4f}  <- production model")
        print(f"FINAL MODEL mAP50-95  : {d['test_mAP50_95']:.4f}")
    print(f"\nSummary JSON : {summary_path}")
    print(f"Models dir   : {work / 'runs'}")
    return 0
if __name__ == "__main__":
    multiprocessing.freeze_support()
    _rc = main()
    try:
        sys.exit(_rc)
    except SystemExit as _e:
        if _e.code:
            raise